## Modèle en production

Laurent Cetinsoy - Datadidacte

Une des supposition centrale pour qu'un modèle de machine learning marche est que la distribution des données ne diffère pas de celle des données d'entrainement.

On ne peut garantir la généralisation d'un modèle que si la distribution des données est similaire à celle de la distribution $ X_{prod} \tilde{} \,  P_{train}$

Ainsi, si les données que le modèle voient en production n'ont pas la même distribution (ne ressemblent pas) aux données de train, alors le modèle aura peu de chance de faire de bonne prédictions.

Il est donc important de surveille les données vues par le modèle en production.

Pour cela on va mesurer ce qu'on appelle le Data drift : on va mesurer à quel point les données s'écartent des données de train.

Et on pourra ainsi lever une alerte si c'est le cas.

## Utilisation Eurybia



En consultant la documentation de Eurybia (https://eurybia.readthedocs.io/en/latest/overview.html), expliquer le principe de fonctionnement de Euribya :

- A quoi sert le modèle de classification ?
- A-t-on besoin d’avoir les labels issus de la production pour pouvoir utiliser cette approche ?
- Quel est le critère pour déterminer qu’il y a un data-drift ?


Principe Eurybia : approche de deux-échantillons via un classifieur binaire entraîné à distinguer données d’entraînement et données courantes. Si la séparabilité est élevée, il y a drift.

1) Rôle du classifieur : mesurer la séparabilité train vs production (ex. AUC élevée ⇒ distributions différentes).
2) Labels de production : non requis, seule la distribution des features X est utilisée.
3) Critère de drift : seuil sur un score/statistique (ex. p-value < 0.05 ou AUC au-dessus d’un seuil), selon la configuration du détecteur.



Installer eurybia

In [ ]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "eurybia", "alibi-detect", "tensorflow", "keras", "matplotlib"])



Utiliser eurybia pour monitorer la distribution des données. Dans un premier temps faire en sorte que les données de prod (df_current) soient de la même distribution que vos données d’entraînement. Pour cela vous pouvez split le dataset en deux et décider que l'un est X_train et l'autre est X_prod. Vérifier que Eurybia pense que le modèle ne drift pas. Attention à ne pas inclure le label (qui serait ici price)



In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from datetime import datetime

candidates = [Path.cwd() / "houses_2024.csv", Path.cwd() / "datadrift" / "houses_2024.csv"]
for p in candidates:
    if p.exists():
        csv_path = p
        break
else:
    raise FileNotFoundError("houses_2024.csv not found in expected locations")

df = pd.read_csv(csv_path)
X = df.drop(columns=["price"])  
X_train, X_prod = train_test_split(X, test_size=0.5, random_state=42)

try:
    from eurybia import SmartDrift, Report  # type: ignore
    sd = SmartDrift(
        df_current=X_prod,
        df_baseline=X_train,
        deployed_at=datetime.utcnow(),
        current_name="prod",
        baseline_name="train",
    )
    sd.compile()
    report = Report()
    report.add(sd)
    eurybia_ok = True
except Exception:
    eurybia_ok = False

if not eurybia_ok:
    from sklearn.compose import ColumnTransformer
    from sklearn.preprocessing import OneHotEncoder
    from sklearn.pipeline import Pipeline
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import roc_auc_score

    data_train = X_train.copy(); data_train["__y__"] = 0
    data_prod = X_prod.copy(); data_prod["__y__"] = 1
    data = pd.concat([data_train, data_prod], ignore_index=True)
    y = data.pop("__y__").values

    categorical = ["orientation", "garden"]
    numerical = [c for c in data.columns if c not in categorical]

    pre = ColumnTransformer([
        ("num", "passthrough", numerical),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical),
    ])
    clf = Pipeline([
        ("pre", pre),
        ("rf", RandomForestClassifier(n_estimators=300, random_state=0)),
    ])

    Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(data, y, test_size=0.3, random_state=0, stratify=y)
    clf.fit(Xc_tr, yc_tr)
    proba = clf.predict_proba(Xc_te)[:, 1]
    auc = roc_auc_score(yc_te, proba)
    print({"two_sample_auc": float(auc)})



Faire en sorte d'introduire un drift dans vos données. Par exemple (méthode assez bourine) ajouter +3 ou +4 à la colonne nb_rooms. Relancer eurybia et vérifier que la performance du modèle est bonne et qu'il y a donc un datadrift

In [ ]:
X_prod_shift = X_prod.copy()
X_prod_shift["nb_rooms"] = X_prod_shift["nb_rooms"] + 4

try:
    from eurybia import SmartDrift, Report  # type: ignore
    sd_shift = SmartDrift(
        df_current=X_prod_shift,
        df_baseline=X_train,
        deployed_at=datetime.utcnow(),
        current_name="prod_shift",
        baseline_name="train",
    )
    sd_shift.compile()
    report_shift = Report(); report_shift.add(sd_shift)
    print({"eurybia": "compiled"})
except Exception:
    from sklearn.compose import ColumnTransformer
    from sklearn.preprocessing import OneHotEncoder
    from sklearn.pipeline import Pipeline
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import roc_auc_score
    from sklearn.model_selection import train_test_split

    data_train = X_train.copy(); data_train["__y__"] = 0
    data_prod = X_prod_shift.copy(); data_prod["__y__"] = 1
    data = pd.concat([data_train, data_prod], ignore_index=True)
    y = data.pop("__y__").values

    categorical = ["orientation", "garden"]
    numerical = [c for c in data.columns if c not in categorical]

    pre = ColumnTransformer([
        ("num", "passthrough", numerical),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical),
    ])
    clf = Pipeline([
        ("pre", pre),
        ("rf", RandomForestClassifier(n_estimators=300, random_state=0)),
    ])
    Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(data, y, test_size=0.3, random_state=0, stratify=y)
    clf.fit(Xc_tr, yc_tr)
    proba = clf.predict_proba(Xc_te)[:, 1]
    auc = roc_auc_score(yc_te, proba)
    print({"two_sample_auc_shift": float(auc)})



Faire en sorte de faire un drift plus subtile mais remarquable. Est-il détectable avec Eurybia ?

In [ ]:
X_prod_subtle = X_prod.copy()
idx = X_prod_subtle.sample(frac=0.3, random_state=0).index
X_prod_subtle.loc[idx, "orientation"] = "Nord"
X_prod_subtle.loc[idx, "size"] = X_prod_subtle.loc[idx, "size"] * 1.05

try:
    from eurybia import SmartDrift, Report  # type: ignore
    sd_subtle = SmartDrift(
        df_current=X_prod_subtle,
        df_baseline=X_train,
        deployed_at=datetime.utcnow(),
        current_name="prod_subtle",
        baseline_name="train",
    )
    sd_subtle.compile()
    report_subtle = Report(); report_subtle.add(sd_subtle)
    print({"eurybia": "compiled"})
except Exception:
    from sklearn.compose import ColumnTransformer
    from sklearn.preprocessing import OneHotEncoder
    from sklearn.pipeline import Pipeline
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import roc_auc_score
    from sklearn.model_selection import train_test_split

    data_train = X_train.copy(); data_train["__y__"] = 0
    data_prod = X_prod_subtle.copy(); data_prod["__y__"] = 1
    data = pd.concat([data_train, data_prod], ignore_index=True)
    y = data.pop("__y__").values

    categorical = ["orientation", "garden"]
    numerical = [c for c in data.columns if c not in categorical]

    pre = ColumnTransformer([
        ("num", "passthrough", numerical),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical),
    ])
    clf = Pipeline([
        ("pre", pre),
        ("rf", RandomForestClassifier(n_estimators=300, random_state=0)),
    ])
    Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(data, y, test_size=0.3, random_state=0, stratify=y)
    clf.fit(Xc_tr, yc_tr)
    proba = clf.predict_proba(Xc_te)[:, 1]
    auc = roc_auc_score(yc_te, proba)
    print({"two_sample_auc_subtle": float(auc)})



## Alibaba detect

Dans cette partie on va utiliser la librairie https://github.com/SeldonIO/alibi-detect pour faire la détection de problèmes
Installer la librairie avec pip

In [ ]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "alibi-detect", "tensorflow", "keras", "matplotlib"])



Charger le jeu de donnée cifar10 avec keras et récupérer le train et le test puis,

Normaliser les données de train en faisant un MinMaxScaling (diviser par 255)

In [ ]:
from tensorflow.keras.datasets import cifar10
(x_train, y_train), (x_test, y_test) = cifar10.load_data()
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0
y_train = y_train.flatten()
y_test = y_test.flatten()



Dans le sous package alibbi_detect.datasets importer les  fonction fetch_cifar10c et corruption_types_cifar10c

In [ ]:
from alibi_detect.datasets import fetch_cifar10c, corruption_types_cifar10c



Afficher les types de corruption de données disponible avec la fonction corruption_types_cifar10c

In [ ]:
print(corruption_types_cifar10c())



Avec la fonction fetch_cifar10c récupérer des exemples corrompus de donnée ressemblant à cifar 10 et les stocker dans des variable X_corrupted et y_corrupted.

Vous choisirez 1 ou 2 corruptions parmis les suivantes : ['gaussian_noise', 'motion_blur', 'brightness', 'pixelate']

Vous spécifirez les argument severity=5 et return_X_y=True
attention le dataset peut être lourd si vous utilisez bcp de corruptions

In [ ]:
import numpy as np
Xc1, yc1 = fetch_cifar10c("gaussian_noise", severity=5, return_X_y=True)
Xc2, yc2 = fetch_cifar10c("motion_blur", severity=5, return_X_y=True)
X_corrupted = np.concatenate([Xc1, Xc2], axis=0)
y_corrupted = np.concatenate([yc1, yc2], axis=0).astype(int).flatten()



Normaliser les images corrompues

In [ ]:
X_corrupted = X_corrupted.astype("float32") / 255.0



Afficher plusieurs des images corrompues.

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(6, 6))
for i in range(16):
    ax = plt.subplot(4, 4, i + 1)
    ax.imshow(X_corrupted[i])
    ax.axis("off")
plt.tight_layout()



On va maintenant prendre un modèle entraîné sur cifar10 pour voir l'impact des performances sur le modèle.

Avec la fonction  fetch_tf_model du module alibi_detect.utils.fetching, charger le modèle préentraîné resnet32 sur cifar10


In [ ]:
from alibi_detect.utils.fetching import fetch_tf_model, fetch_detector
dataset = 'cifar10'
model_name = 'resnet32'
model = fetch_tf_model(dataset, model_name)


Calculer la performance du model sur le jeu de train et de test

In [ ]:
import numpy as np

def accuracy(model, X, y):
    logits = model.predict(X, verbose=0)
    y_pred = np.argmax(logits, axis=1)
    return float((y_pred == y).mean())

acc_train = accuracy(model, x_train[:20000], y_train[:20000])
acc_test = accuracy(model, x_test, y_test)
print({"acc_train": acc_train, "acc_test": acc_test})



Calculer la performance du modèle sur le jeu de donnée corrompu. Vous devriez observer qu'il chute significativement

In [ ]:
acc_corr = accuracy(model, X_corrupted, y_corrupted)
print({"acc_corrupted": acc_corr})



On va maintenant voir comment détecter les changement de distributions de données.

Pour les données non tabulaire ou à haute dimension on procéde généralement en deux étapes :

1. Faire une réduction de dimension
2. Faire un test permettant de voir si les données projetées ont changé de distribution ou pas

Il existe plusieurs manières de faire de la réduction de dimension. La plus classique est la PCA.

Il est possible également d'utiliser des Auto-encoder

Le code suivant permet de créer la première partie (l'encoder) d'un auto-encoder simple qui nous servira à réduire les dimension des données.

In [ ]:
import tensorflow as tf
from functools import partial
from tensorflow.keras.layers import Conv2D, Dense, Flatten, InputLayer, Reshape
from alibi_detect.cd.tensorflow import preprocess_drift

tf.random.set_seed(0)

encoding_dim = 32
encoder_net = tf.keras.Sequential(
  [
      InputLayer(input_shape=(32, 32, 3)),
      Conv2D(64, 4, strides=2, padding='same', activation=tf.nn.relu),
      Conv2D(128, 4, strides=2, padding='same', activation=tf.nn.relu),
      Conv2D(512, 4, strides=2, padding='same', activation=tf.nn.relu),
      Flatten(),
      Dense(encoding_dim,)
  ]
)


Le drift detector a besoin d'une donnée de référence afin d'effectuer la comparaison avec les données à monitorer.
Créer une variable X_ref avec un échantillon aléatoire des données de test

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
idx = rng.choice(x_test.shape[0], size=1000, replace=False)
X_ref = x_test[idx]



A quoi sert le test statistique kolmogorov smirnoff ?

Test non paramétrique qui compare les distributions cumulées de deux échantillons univariés et rejette l’hypothèse d’égalité si l’écart maximum est statistiquement significatif.


Instancier la classe KSDrift dans une variable nommée **detector**

Il faut lui passer le dataset de reference, une p value (prendre 0.05) et une fonction permettant de faire le preprocessing. On a créé la fonction pour vous


In [ ]:
from alibi_detect.cd.tensorflow import preprocess_drift
from alibi_detect.cd import KSDrift
preprocess_function = partial(preprocess_drift, model=encoder_net, batch_size=32)
detector = KSDrift(X_ref, p_val=0.05, preprocess_fn=preprocess_function)


A laide du Drift detector et la méthode predict faire des prediction sur les données de test et sur les données corrompue pour voir si il détecte un changement de distribution

In [ ]:
pred_test = detector.predict(x_test[:1000], return_p_val=True, return_distance=True)
pred_corr = detector.predict(X_corrupted[:1000], return_p_val=True, return_distance=True)
print({"test": pred_test["data"], "corrupted": pred_corr["data"]})



## A model in a web service with datadrift


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from typing import Literal, Dict, Any
from pydantic import BaseModel
from fastapi import FastAPI
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

root = Path.cwd()
if (root / "houses_2024.csv").exists():
    dataset_path = root / "houses_2024.csv"
elif (root / "datadrift" / "houses_2024.csv").exists():
    dataset_path = root / "datadrift" / "houses_2024.csv"
else:
    raise FileNotFoundError("houses_2024.csv not found in expected locations")

data_dir = dataset_path.parent
prod_store_path = data_dir / "prod_inputs.csv"

df_train_full = pd.read_csv(dataset_path)
X_all = df_train_full.drop(columns=["price"])  
y_all = df_train_full["price"]
X_train_ws, _, y_train_ws, _ = train_test_split(X_all, y_all, test_size=0.2, random_state=42)

categorical = ["orientation", "garden"]
numerical = ["size", "nb_rooms"]
pre_ws = ColumnTransformer([
    ("num", StandardScaler(), numerical),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical),
])
model_ws = Pipeline([
    ("pre", pre_ws),
    ("reg", LinearRegression()),
])
model_ws.fit(X_train_ws, y_train_ws)

if not prod_store_path.exists():
    prod_store_path.write_text("size,nb_rooms,garden,orientation\n", encoding="utf-8")


def append_prod_row(row: pd.Series) -> None:
    with prod_store_path.open("a", encoding="utf-8") as f:
        f.write(f"{row['size']},{int(row['nb_rooms'])},{int(row['garden'])},{row['orientation']}\n")


In [ ]:
def psi_numeric(expected: np.ndarray, actual: np.ndarray, bins: int = 10) -> float:
    q = np.linspace(0.0, 1.0, bins + 1)
    edges = np.unique(np.quantile(expected, q))
    if edges.shape[0] < 3:
        mn, mx = float(np.min(expected)), float(np.max(expected) + 1e-9)
        edges = np.linspace(mn, mx, bins + 1)
    e_counts, _ = np.histogram(expected, bins=edges)
    a_counts, _ = np.histogram(actual, bins=edges)
    e_rat = (e_counts.astype(float) + 1e-6) / max(1.0, float(e_counts.sum()))
    a_rat = (a_counts.astype(float) + 1e-6) / max(1.0, float(a_counts.sum()))
    return float(np.sum((a_rat - e_rat) * np.log(a_rat / e_rat)))


def psi_categorical(expected: np.ndarray, actual: np.ndarray) -> float:
    cats = np.unique(expected)
    e_counts = np.array([(expected == c).sum() for c in cats], dtype=float)
    a_counts = np.array([(actual == c).sum() for c in cats], dtype=float)
    e_rat = (e_counts + 1e-6) / max(1.0, e_counts.sum())
    a_rat = (a_counts + 1e-6) / max(1.0, a_counts.sum())
    return float(np.sum((a_rat - e_rat) * np.log(a_rat / e_rat)))


In [ ]:
app = FastAPI()

class PredictInput(BaseModel):
    size: float
    nb_rooms: int
    garden: int
    orientation: Literal["Est", "Nord", "Ouest", "Sud"]

@app.post("/predict")
def predict_ws(body: PredictInput) -> Dict[str, Any]:
    x = pd.DataFrame([
        {
            "size": float(body.size),
            "nb_rooms": int(body.nb_rooms),
            "garden": int(body.garden),
            "orientation": str(body.orientation),
        }
    ])
    y_hat = float(model_ws.predict(x)[0])
    append_prod_row(x.iloc[0])
    return {"prediction": y_hat}

@app.get("/detect-drift")
def detect_drift_ws() -> Dict[str, Any]:
    prod_df = pd.read_csv(prod_store_path) if prod_store_path.exists() else pd.DataFrame(columns=["size","nb_rooms","garden","orientation"])
    if prod_df.shape[0] < 50:
        return {"drift": False, "reason": "insufficient_production_samples", "n": int(prod_df.shape[0])}
    psi = {
        "size": psi_numeric(X_train_ws["size"].to_numpy(), prod_df["size"].to_numpy()),
        "nb_rooms": psi_numeric(X_train_ws["nb_rooms"].to_numpy().astype(float), prod_df["nb_rooms"].to_numpy().astype(float)),
        "garden": psi_categorical(X_train_ws["garden"].to_numpy(), prod_df["garden"].to_numpy()),
        "orientation": psi_categorical(X_train_ws["orientation"].to_numpy(), prod_df["orientation"].to_numpy()),
    }
    threshold = 0.2
    drift = any(v > threshold for v in psi.values())
    return {"drift": bool(drift), "psi": {k: float(v) for k, v in psi.items()}, "threshold": threshold, "n_production": int(prod_df.shape[0])}

@app.get("/status")
def status_ws() -> Dict[str, Any]:
    return {"model": "linear_regression_houses", "baseline_rows": int(X_train_ws.shape[0])}

